# P00b — Laeven & Valencia Systemic Crisis Database

Load and explore the IMF Systemic Banking Crises Database. Convert to Arrow.

**Source:** [Laeven & Valencia (2018)](https://www.imf.org/en/publications/wp/issues/2018/09/14/systemic-banking-crises-revisited-46232) — IMF WP/18/206. Coverage: 1970–2017, ~190 countries.

**Sheets:**
- *Crisis Years* — country × crisis type (banking, currency, sovereign debt)
- *Crisis Resolution and Outcomes* — fiscal costs, output losses, NPLs, public debt increase

**Role:** Tier 1 backtesting. Discrete regime-shift markers — when the model identifies a country as approaching super-critical, do crisis dates align?

In [1]:
include("phase00b/functions/load_phase00b.jl")

## 1. Load Crisis Years

In [2]:
crises = lv_load_crisis_years()
println("$(nrow(crises)) countries × $(ncol(crises)) cols")
describe(crises)

165 countries × 5 cols


Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Nothing,Union…,Nothing,Union…,Int64,Type
1,country,,Albania,,Zimbabwe,0,"Union{Missing, String}"
2,banking_crisis_year,,,,,47,Any
3,currency_crisis_year,,,,,46,Any
4,sovereign_debt_crisis_year,,,,,103,Any
5,sovereign_debt_restructuring_year,,,,,102,Any


In [3]:
first(crises, 10)

Row,country,banking_crisis_year,currency_crisis_year,sovereign_debt_crisis_year,sovereign_debt_restructuring_year
,String?,Any,Any,Any,Any
1,Albania,1994,1997,1990,1992
2,Algeria,1990,"1988, 1994",missing,missing
3,Angola,missing,"1991, 1996, 2015",1988,1992
4,Argentina,"1980, 1989, 1995, 2001","1975, 1981, 1987, 2002, 2013","1982, 2001, 2014","1993, 2005, 2016"
5,Armenia,1994,missing,missing,missing
6,Australia,missing,missing,missing,missing
7,Austria,2008,missing,missing,missing
8,Azerbaijan,1995,2015,missing,missing
9,Bangladesh,1987,1976,missing,missing


In [4]:
# What do the crisis year values look like?
println("Sample banking_crisis_year values:")
for v in first(skipmissing(crises.banking_crisis_year), 15)
    println("  $(repr(v)) ($(typeof(v)))")
end
println()
println("Sample sovereign_debt_crisis_year values:")
for v in first(skipmissing(crises.sovereign_debt_crisis_year), 10)
    println("  $(repr(v)) ($(typeof(v)))")
end


Sample banking_crisis_year values:
  1994 (Int64)
  1990 (Int64)
  "1980, 1989, 1995, 2001" (String)
  1994 (Int64)
  2008 (Int64)
  1995 (Int64)
  1987 (Int64)
  1995 (Int64)
  2008 (Int64)
  1988 (Int64)
  "1986, 1994" (String)
  1992 (Int64)
  "1990, 1994" (String)
  1996 (Int64)
  1990 (Int64)

Sample sovereign_debt_crisis_year values:
  1990 (Int64)
  1988 (Int64)
  "1982, 2001, 2014" (String)
  "2007, 2012, 2017" (String)
  1980 (Int64)
  1983 (Int64)
  1990 (Int64)
  1989 (Int64)
  1983 (Int64)
  1976 (Int64)


In [5]:
# Parse all crisis years — how many unique events?
function parse_years(val)
    ismissing(val) && return Int[]
    if val isa Number
        return [Int(val)]
    end
    s = string(val)
    years = Int[]
    for part in split(s, ",")
        p = strip(part)
        y = tryparse(Int, p)
        !isnothing(y) && push!(years, y)
    end
    return years
end

# Count events by type
for col in [:banking_crisis_year, :currency_crisis_year, :sovereign_debt_crisis_year, :sovereign_debt_restructuring_year]
    all_years = vcat([parse_years(v) for v in crises[!, col]]...)
    println("$(rpad(string(col), 38)) $(length(all_years)) events, $(length(unique(all_years))) unique years ($(isempty(all_years) ? 0 : minimum(all_years))–$(isempty(all_years) ? 0 : maximum(all_years)))")
end


banking_crisis_year                    151 events, 30 unique years (1976–2014)
currency_crisis_year                   239 events, 43 unique years (1971–2016)
sovereign_debt_crisis_year             76 events, 29 unique years (1976–2017)
sovereign_debt_restructuring_year      75 events, 27 unique years (1982–2017)


In [6]:
# Explode to country-year panel with binary crisis flags
rows = []
for r in eachrow(crises)
    country = r.country
    for yr in union(
        parse_years(r.banking_crisis_year),
        parse_years(r.currency_crisis_year),
        parse_years(r.sovereign_debt_crisis_year),
        parse_years(r.sovereign_debt_restructuring_year)
    )
        push!(rows, (
            country = country,
            year = yr,
            banking = yr in parse_years(r.banking_crisis_year),
            currency = yr in parse_years(r.currency_crisis_year),
            sovereign_debt = yr in parse_years(r.sovereign_debt_crisis_year),
            debt_restructuring = yr in parse_years(r.sovereign_debt_restructuring_year)
        ))
    end
end

lv_panel = DataFrame(rows)
sort!(lv_panel, [:country, :year])
println("Panel: $(nrow(lv_panel)) country-year-crisis rows")
println("Countries: $(length(unique(lv_panel.country)))")
println("Years: $(minimum(lv_panel.year))–$(maximum(lv_panel.year))")
println()
println("Crisis counts:")
println("  Banking:           $(count(lv_panel.banking))")
println("  Currency:          $(count(lv_panel.currency))")
println("  Sovereign debt:    $(count(lv_panel.sovereign_debt))")
println("  Debt restructure:  $(count(lv_panel.debt_restructuring))")
println()
# Twin/triple crises
lv_panel[!, :n_crises] = lv_panel.banking .+ lv_panel.currency .+ lv_panel.sovereign_debt
println("  Twin crisis (2+):  $(count(>=(2), lv_panel.n_crises))")
println("  Triple crisis (3): $(count(==(3), lv_panel.n_crises))")
println()
first(lv_panel, 10)


Panel: 488 country-year-crisis rows
Countries: 157
Years: 1971–2017

Crisis counts:
  Banking:           151
  Currency:          239
  Sovereign debt:    76
  Debt restructure:  75

  Twin crisis (2+):  31
  Triple crisis (3): 7



Row,country,year,banking,currency,sovereign_debt,debt_restructuring,n_crises
,String,Int64,Bool,Bool,Bool,Bool,Int64
1,Albania,1990,false,false,true,false,1
2,Albania,1992,false,false,false,true,0
3,Albania,1994,true,false,false,false,1
4,Albania,1997,false,true,false,false,1
5,Algeria,1988,false,true,false,false,1
6,Algeria,1990,true,false,false,false,1
7,Algeria,1994,false,true,false,false,1
8,Angola,1988,false,false,true,false,1
9,Angola,1991,false,true,false,false,1


## 2. Load Outcomes

In [7]:
outcomes = lv_load_outcomes()
println("$(nrow(outcomes)) crisis episodes × $(ncol(outcomes)) cols")
describe(outcomes)

152 crisis episodes × 11 cols


Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Union…,Any,Union…,Any,Int64,Type
1,country,,"1/ In percent of GDP. Output losses are computed as the cumulative sum of the differences between actual and trend real GDP over the period [T, T+3], expressed in percent of trend real GDP, with T denoting the starting year of the crisis. The trend is computed by applying an HP filter (λ=100) to the GDP series over [T-20, T-1]. No output losses are reported for crises in transition economies that took place during the period of transition to market economies.\r\n2/ Fiscal costs refer to outlays directly related to the restructuring of the financial sector. \r\n3/ Liquidity is measured as the ratio of central bank claims on deposit money banks (line 12 in IFS) and liquidity support from the Treasury to total deposits and liabilities to non-residents. Total deposits are computed as the sum of demand deposits (line 24), other deposits (line 25), and liabilities to non-residents (line 26).\r\n4/ In percent of total loans.\r\n5/ In percent of GDP. For episodes starting in 2007 and later, the increase in public debt is measured as the change in debt projections, over [T-1, T+3], relative to the pre-crisis debt projections, where T is the starting year of the crisis.\r\n6/ Credit data missing. For these countries, end dates are based on GDP growth only. \r\n7/ We truncate the duration of crises at 5 years, starting with the first crisis year. \r\n8/ Borderline cases.\r\nSource: WEO, IFS, IMF Staff reports, IMF Financial Soundness Indicators, Laeven and Valencia (2013), and authors’ calculation.\r\n",,Zimbabwe,0,String
2,start_year,1994.25,1976,1994.0,2014,1,"Union{Missing, Int64}"
3,end_year,,,,,1,Any
4,output_loss_pct,,,,,2,Any
5,fiscal_cost_pct_gdp,,,,,1,Any
6,fiscal_cost_net_pct_gdp,,,,,1,Any
7,fiscal_cost_pct_fin_assets,,,,,1,Any
8,peak_liquidity_pct,,,,,1,Any
9,liquidity_support_pct,,,,,1,Any


In [8]:
first(outcomes, 10)

Row,country,start_year,end_year,output_loss_pct,fiscal_cost_pct_gdp,fiscal_cost_net_pct_gdp,fiscal_cost_pct_fin_assets,peak_liquidity_pct,liquidity_support_pct,peak_npls_pct,public_debt_increase_pct
,String,Int64?,Any,Any,Any,Any,Any,Any,Any,Any,Any
1,Albania,1994,1994,…,...,...,...,7.61596,...,26.8,...
2,Algeria,1990,1994 7/,41.3923,...,...,...,37.5603,29.8525,30.0,19.1006
3,Argentina,1980,1982 6/,58.1712,55.1,55.1,213.898,64.6459,62.2123,9.0,33.1474
4,Argentina,1989,1991,12.6299,6.0,6.0,21.606,151.572,135.665,27.0,-21.2926
5,Argentina 8/,1995,1995,0.0,2.0,2.0,8.55066,71.3528,62.9977,17.0,8.69467
6,Argentina,2001,2003,70.9742,9.6,9.58466,28.0948,22.8584,22.6466,20.1,81.8872
7,Armenia 4/,1994,1994 6/,…,...,...,...,41.3769,22.994,...,...
8,Austria,2008,2012 7/,19.2159,5.15,1.64,1.64353,10.02,6.41,4.1,19.8
9,Azerbaijan,1995,1995 6/,...,...,...,...,127.58,84.4638,...,0.89603


In [9]:
# What does the outcomes sheet look like?
println("$(nrow(outcomes)) rows × $(ncol(outcomes)) cols")
println("Columns: $(names(outcomes))")
println()
println("Sample rows:")
first(outcomes, 15)


152 rows × 11 cols
Columns: ["country", "start_year", "end_year", "output_loss_pct", "fiscal_cost_pct_gdp", "fiscal_cost_net_pct_gdp", "fiscal_cost_pct_fin_assets", "peak_liquidity_pct", "liquidity_support_pct", "peak_npls_pct", "public_debt_increase_pct"]

Sample rows:


Row,country,start_year,end_year,output_loss_pct,fiscal_cost_pct_gdp,fiscal_cost_net_pct_gdp,fiscal_cost_pct_fin_assets,peak_liquidity_pct,liquidity_support_pct,peak_npls_pct,public_debt_increase_pct
,String,Int64?,Any,Any,Any,Any,Any,Any,Any,Any,Any
1,Albania,1994,1994,…,...,...,...,7.61596,...,26.8,...
2,Algeria,1990,1994 7/,41.3923,...,...,...,37.5603,29.8525,30.0,19.1006
3,Argentina,1980,1982 6/,58.1712,55.1,55.1,213.898,64.6459,62.2123,9.0,33.1474
4,Argentina,1989,1991,12.6299,6.0,6.0,21.606,151.572,135.665,27.0,-21.2926
5,Argentina 8/,1995,1995,0.0,2.0,2.0,8.55066,71.3528,62.9977,17.0,8.69467
6,Argentina,2001,2003,70.9742,9.6,9.58466,28.0948,22.8584,22.6466,20.1,81.8872
7,Armenia 4/,1994,1994 6/,…,...,...,...,41.3769,22.994,...,...
8,Austria,2008,2012 7/,19.2159,5.15,1.64,1.64353,10.02,6.41,4.1,19.8
9,Azerbaijan,1995,1995 6/,...,...,...,...,127.58,84.4638,...,0.89603


In [10]:
# Clean outcomes data
# 1. Strip footnote annotations from country names ("Argentina 8/" → "Argentina")
outcomes[!, :country_clean] = [ismissing(v) ? missing : replace(String(v), r"\s*\d+/\s*$" => "") for v in outcomes.country]

# 2. Clean end_year — strip footnote annotations and parse to Int
outcomes[!, :end_year_clean] = map(outcomes.end_year) do v
    ismissing(v) && return missing
    s = string(v)
    s = replace(s, r"\s*\d+/\s*$" => "")  # strip "7/" etc
    y = tryparse(Int, strip(s))
    return y
end

# 3. Drop the footnotes row (country contains a very long string)
filter!(r -> !ismissing(r.start_year), outcomes)

# Show cleaned
println("$(nrow(outcomes)) episodes after cleanup")
first(select(outcomes, :country_clean, :start_year, :end_year, :end_year_clean, :output_loss_pct), 15)


151 episodes after cleanup


Row,country_clean,start_year,end_year,end_year_clean,output_loss_pct
,String,Int64?,Any,Union…?,Any
1,Albania,1994,1994,1994,…
2,Algeria,1990,1994 7/,1994,41.3923
3,Argentina,1980,1982 6/,1982,58.1712
4,Argentina,1989,1991,1991,12.6299
5,Argentina,1995,1995,1995,0.0
6,Argentina,2001,2003,2003,70.9742
7,Armenia,1994,1994 6/,1994,…
8,Austria,2008,2012 7/,2012,19.2159
9,Azerbaijan,1995,1995 6/,1995,...


In [11]:
# Cluster crises by start year — which years had the most simultaneous banking crises?
crisis_by_year = combine(groupby(outcomes, :start_year),
    nrow => :n_episodes,
    :country_clean => (x -> join(x, ", ")) => :countries
)
sort!(crisis_by_year, :n_episodes, rev=true)
println("Crisis clustering by start year:")
first(crisis_by_year, 15)


Crisis clustering by start year:


Row,start_year,n_episodes,countries
,Int64?,Int64,String
1,2008,22,"Austria, Belgium, Denmark, France, Germany, Greece, Hungary, Iceland, Ireland, Italy, Kazakhstan, Latvia, Luxembourg, Mongolia, Netherlands, Portugal , Russia, Slovenia , Spain, Sweden, Switzerland, Ukraine"
2,1995,13,"Argentina, Azerbaijan , Belarus , Cameroon, Central African Rep, Guinea-Bissau, Kyrgyz Rep , Latvia , Lithuania , Paraguay, Swaziland, Zambia, Zimbabwe"
3,1994,11,"Albania, Armenia, Bolivia, Brazil, Burundi, Congo, Dem Rep, Costa Rica, Haiti, Mexico, Uganda, Venezuela"
4,1991,10,"Congo, Dem Rep, Djibouti, Finland, Georgia , Hungary , Liberia, Nigeria, Norway, Sweden, Tunisia"
5,1983,8,"Chad, Congo, Dem Rep, Equatorial Guinea, Israel, Niger, Peru, Philippines, Thailand"
6,1992,8,"Bosnia and Herzegovina , Chad, Congo, Rep, Estonia , Kenya, Poland , São Tomé & Príncipe, Slovenia"
7,1998,8,"China, Mainland, Colombia, Croatia , Ecuador, Romania , Russia , Slovak Rep, Ukraine"
8,1988,7,"Benin, Cote d'Ivoire, Madagascar, Nepal, Panama, Senegal, United States"
9,1993,7,"Cape Verde, Eritrea, Guinea, Guyana, India, Macedonia, FYR , Togo"


In [13]:
# Build country-year panel — flag every year a crisis was active [start, end]
crisis_panel_rows = []
for r in eachrow(outcomes)
    ismissing(r.start_year) && continue
    end_yr = r.end_year_clean
    if isnothing(end_yr) || ismissing(end_yr)
        end_yr = r.start_year  # if no end, crisis lasted 1 year
    end
    for yr in r.start_year:end_yr
        push!(crisis_panel_rows, (
            country = r.country_clean,
            year = yr,
            crisis_start = r.start_year,
            crisis_end = end_yr
        ))
    end
end

crisis_panel = DataFrame(crisis_panel_rows)
sort!(crisis_panel, [:country, :year])
println("Crisis-active country-years: $(nrow(crisis_panel))")
println("Unique countries: $(length(unique(crisis_panel.country)))")
println("Years: $(minimum(crisis_panel.year))–$(maximum(crisis_panel.year))")
println()

# How many countries in crisis per year?
yr_counts = combine(groupby(crisis_panel, :year), 
    :country => (x -> length(unique(x))) => :n_countries_in_crisis)
sort!(yr_counts, :year)
println("Countries in active banking crisis per year:")
for r in eachrow(yr_counts)
    bar = "█" ^ min(r.n_countries_in_crisis, 50)
    println("  $(r.year): $(rpad(bar, 52)) $(r.n_countries_in_crisis)")
end


Crisis-active country-years: 463
Unique countries: 123
Years: 1976–2015

Countries in active banking crisis per year:
  1976: ██                                                   2
  1977: █                                                    1
  1978: █                                                    1
  1979: █                                                    1
  1980: ████                                                 4
  1981: ██████                                               6
  1982: ██████████                                           10
  1983: ████████████████                                     16
  1984: ███████████                                          11
  1985: ██████████                                           10
  1986: ████                                                 4
  1987: ██████                                               6
  1988: ████████████                                         12
  1989: ████████████                                      

In [14]:
# QoG alignment — these are country names, not ISO3 codes
# We need to map country names to ISO3
println("Sample country names:")
for c in sort(unique(crisis_panel.country))[1:20]
    println("  $c")
end
println("  ...")
println("Total unique countries: $(length(unique(crisis_panel.country)))")


Sample country names:
  Albania
  Algeria
  Argentina
  Armenia
  Austria
  Azerbaijan 
  Bangladesh
  Belarus 
  Belgium
  Benin
  Bolivia
  Bosnia and Herzegovina 
  Brazil
  Bulgaria
  Burkina Faso
  Burundi
  Cameroon
  Cape Verde
  Central African Rep
  Chad
  ...
Total unique countries: 123


In [15]:
# Build country name → ISO3 mapping from QoG
qog = DataFrame(Arrow.Table("data/qog_std_ts_jan25_aug.arrow"))
qog_names = unique(select(dropmissing(qog, [:ident_cname, :ident_ccodealp]), :ident_cname, :ident_ccodealp))

# Also get CEPII Gravity country names as a backup
grav_countries = DataFrame(Arrow.Table("data/cepii_gravity_countries.arrow"))

# Strip trailing whitespace from L&V country names
crisis_panel[!, :country] = strip.(crisis_panel.country)
outcomes[!, :country_clean] = strip.(outcomes.country_clean)

# Try matching L&V names to QoG names
lv_countries = sort(unique(crisis_panel.country))
matched = String[]
unmatched = String[]

name_to_iso3 = Dict{String, String}()
for c in lv_countries
    # Exact match
    row = filter(r -> r.ident_cname == c, qog_names)
    if nrow(row) > 0
        name_to_iso3[c] = first(row.ident_ccodealp)
        push!(matched, c)
    else
        push!(unmatched, c)
    end
end

println("Matched: $(length(matched)) / $(length(lv_countries))")
println("\nUnmatched ($(length(unmatched))):")
for c in unmatched
    println("  \"$c\"")
end


Matched: 93 / 118

Unmatched (25):
  "Bolivia"
  "Cape Verde"
  "Central African Rep"
  "China, Mainland"
  "Congo, Dem Rep"
  "Congo, Rep"
  "Cote d'Ivoire"
  "Czech Republic"
  "Dominican Rep"
  "Korea"
  "Kyrgyz Rep"
  "Macedonia, FYR"
  "Moldova"
  "Netherlands"
  "Niger"
  "Philippines"
  "Russia"
  "Slovak Rep"
  "Swaziland"
  "São Tomé & Príncipe"
  "Tanzania"
  "United Kingdom"
  "United States"
  "Venezuela"
  "Vietnam"


In [16]:
# Manual name mapping for unmatched countries
lv_name_remap = Dict(
    "Bolivia" => "BOL",
    "Cape Verde" => "CPV",
    "Central African Rep" => "CAF",
    "China, Mainland" => "CHN",
    "Congo, Dem Rep" => "COD",
    "Congo, Rep" => "COG",
    "Cote d'Ivoire" => "CIV",
    "Czech Republic" => "CZE",
    "Dominican Rep" => "DOM",
    "Korea" => "KOR",
    "Kyrgyz Rep" => "KGZ",
    "Macedonia, FYR" => "MKD",
    "Moldova" => "MDA",
    "Netherlands" => "NLD",
    "Niger" => "NER",
    "Philippines" => "PHL",
    "Russia" => "RUS",
    "Slovak Rep" => "SVK",
    "Swaziland" => "SWZ",
    "São Tomé & Príncipe" => "STP",
    "Tanzania" => "TZA",
    "United Kingdom" => "GBR",
    "United States" => "USA",
    "Venezuela" => "VEN",
    "Vietnam" => "VNM",
)

# Merge both mappings
merge!(name_to_iso3, lv_name_remap)

# Verify all covered
still_unmatched = [c for c in lv_countries if !haskey(name_to_iso3, c)]
println("Still unmatched: $(length(still_unmatched))")
if !isempty(still_unmatched)
    for c in still_unmatched; println("  \"$c\""); end
end

# Add ISO3 to panel
crisis_panel[!, :iso3] = [get(name_to_iso3, c, missing) for c in crisis_panel.country]
outcomes[!, :iso3] = [get(name_to_iso3, c, missing) for c in outcomes.country_clean]
println("\nPanel rows with ISO3: $(count(!ismissing, crisis_panel.iso3)) / $(nrow(crisis_panel))")


Still unmatched: 0

Panel rows with ISO3: 463 / 463


In [17]:
# Check outcomes cleaning — are the footnote-annotated values parsed correctly?
println("Outcomes columns and types:")
for col in names(outcomes)
    println("  $(rpad(string(col), 30)) $(eltype(outcomes[!, col]))")
end
println()

# The numeric columns are still typed as Any — "..." values weren't converted to missing
println("Sample output_loss_pct values:")
for v in first(outcomes.output_loss_pct, 15)
    println("  $(repr(v)) ($(typeof(v)))")
end


Outcomes columns and types:
  country                        String
  start_year                     Union{Missing, Int64}
  end_year                       Any
  output_loss_pct                Any
  fiscal_cost_pct_gdp            Any
  fiscal_cost_net_pct_gdp        Any
  fiscal_cost_pct_fin_assets     Any
  peak_liquidity_pct             Any
  liquidity_support_pct          Any
  peak_npls_pct                  Any
  public_debt_increase_pct       Any
  country_clean                  SubString{String}
  end_year_clean                 Union{Missing, Nothing, Int64}
  iso3                           String

Sample output_loss_pct values:
  "…" (String)
  41.3922533 (Float64)
  58.17123480000001 (Float64)
  12.6299469 (Float64)
  0.0 (Float64)
  70.9741988 (Float64)
  "…" (String)
  19.215918 (Float64)
  "..." (String)
  0.0 (Float64)
  "..." (String)
  15.732718 (Float64)
  14.9175882 (Float64)
  49.1917352 (Float64)
  0.0 (Float64)


In [18]:
# Clean outcomes — convert "..." and "…" to missing, parse numeric columns
function clean_numeric!(df::DataFrame, col::Symbol)
    df[!, col] = map(df[!, col]) do v
        ismissing(v) && return missing
        v isa Number && return Float64(v)
        s = strip(string(v))
        (s == "..." || s == "…" || s == "" || s == "0") && return missing
        y = tryparse(Float64, s)
        return y  # returns nothing if unparseable
    end
    # Convert nothing to missing
    df[!, col] = [isnothing(v) ? missing : v for v in df[!, col]]
    return df
end

numeric_cols = [:end_year, :output_loss_pct, :fiscal_cost_pct_gdp, :fiscal_cost_net_pct_gdp,
                :fiscal_cost_pct_fin_assets, :peak_liquidity_pct, :liquidity_support_pct,
                :peak_npls_pct, :public_debt_increase_pct]

for col in numeric_cols
    clean_numeric!(outcomes, col)
end

# Drop the intermediate columns, keep clean versions
select!(outcomes, :iso3, :country_clean => :country, :start_year, 
        :end_year_clean => :end_year, :output_loss_pct, :fiscal_cost_pct_gdp,
        :fiscal_cost_net_pct_gdp, :fiscal_cost_pct_fin_assets, :peak_liquidity_pct,
        :liquidity_support_pct, :peak_npls_pct, :public_debt_increase_pct)

println("Cleaned outcomes: $(nrow(outcomes)) rows × $(ncol(outcomes)) cols")
println()
println("Missingness in outcome measures:")
for col in names(outcomes)
    n_miss = count(ismissing, outcomes[!, col])
    n_miss == 0 && continue
    pct = round(100 * n_miss / nrow(outcomes), digits=1)
    println("  $(rpad(string(col), 30)) $pct% missing")
end


Cleaned outcomes: 151 rows × 12 cols

Missingness in outcome measures:
  output_loss_pct                11.9% missing
  fiscal_cost_pct_gdp            40.4% missing
  fiscal_cost_net_pct_gdp        55.0% missing
  fiscal_cost_pct_fin_assets     42.4% missing
  peak_liquidity_pct             3.3% missing
  liquidity_support_pct          9.9% missing
  peak_npls_pct                  27.2% missing
  public_debt_increase_pct       10.6% missing


In [19]:
# Temporal variance — is missingness concentrated in older crises?
outcomes[!, :decade] = div.(outcomes.start_year, 10) .* 10
for col in [:output_loss_pct, :fiscal_cost_pct_gdp, :peak_npls_pct, :public_debt_increase_pct]
    println("$(col):")
    for g in groupby(sort(outcomes, :decade), :decade)
        d = first(g.decade)
        ismissing(d) && continue
        n = nrow(g)
        n_miss = count(ismissing, g[!, col])
        pct = round(100 * n_miss / n, digits=0)
        println("  $(d)s: $(n_miss)/$n missing ($(Int(pct))%)")
    end
    println()
end


output_loss_pct:
  1970s: 0/3 missing (0%)
  1980s: 0/40 missing (0%)
  1990s: 17/74 missing (23%)
  2000s: 0/30 missing (0%)
  2010s: 1/4 missing (25%)

fiscal_cost_pct_gdp:
  1970s: 2/3 missing (67%)
  1980s: 20/40 missing (50%)
  1990s: 38/74 missing (51%)
  2000s: 0/30 missing (0%)
  2010s: 1/4 missing (25%)

peak_npls_pct:
  1970s: 2/3 missing (67%)
  1980s: 18/40 missing (45%)
  1990s: 21/74 missing (28%)
  2000s: 0/30 missing (0%)
  2010s: 0/4 missing (0%)

public_debt_increase_pct:
  1970s: 0/3 missing (0%)
  1980s: 4/40 missing (10%)
  1990s: 12/74 missing (16%)
  2000s: 0/30 missing (0%)
  2010s: 0/4 missing (0%)



In [21]:
# Which countries have the most missing outcome data?
outcomes[!, :n_missing] = [count(col -> ismissing(outcomes[i, col]), 
    [:output_loss_pct, :fiscal_cost_pct_gdp, :peak_npls_pct, :public_debt_increase_pct])
    for i in 1:nrow(outcomes)]

# Show episodes with 3+ missing of the 4 key measures
high_miss = filter(r -> r.n_missing >= 3, outcomes)
sort!(high_miss, :start_year)
println("Episodes with 3+ of 4 key measures missing ($(nrow(high_miss))):")
select(high_miss, :iso3, :country, :start_year, :end_year, :n_missing)


Episodes with 3+ of 4 key measures missing (11):


Row,iso3,country,start_year,end_year,n_missing
,String,SubStrin…,Int64?,Union…?,Int64
1,GNQ,Equatorial Guinea,1983,1983,3
2,LBN,Lebanon,1990,1993,3
3,DJI,Djibouti,1991,1995,3
4,GEO,Georgia,1991,1995,3
5,LBR,Liberia,1991,1995,4
6,BIH,Bosnia and Herzegovina,1992,1996,4
7,ERI,Eritrea,1993,1993,4
8,ALB,Albania,1994,1994,3
9,ARM,Armenia,1994,1994,4


In [22]:
filter(r -> coalesce(div(r.start_year, 10) * 10, 0) == 2010, 
    select(outcomes, :iso3, :country, :start_year, :end_year, 
           :output_loss_pct, :fiscal_cost_pct_gdp, :peak_npls_pct, :public_debt_increase_pct))


Row,iso3,country,start_year,end_year,output_loss_pct,fiscal_cost_pct_gdp,peak_npls_pct,public_debt_increase_pct
,String,SubStrin…,Int64?,Union…?,Float64?,Float64?,Float64?,Float64?
1,CYP,Cyprus,2011,2015,76.4953,17.99,47.75,21.2567
2,GNB,Guinea-Bissau,2014,,0.0,missing,25.7,3.16386
3,MDA,Moldova,2014,,missing,11.7,16.41,19.5212
4,UKR,Ukraine,2014,,93.2281,13.92,55.11,53.4


In [20]:
# Verify cleaning — spot check known crises against published values
# Argentina 2001: output loss ~71%, fiscal cost ~9.6%, NPLs ~20%
# US 2007/2008: output loss ~31%, fiscal cost ~4.5%
# Indonesia 1997: output loss ~69%, fiscal cost ~57%
for (c, yr) in [("Argentina", 2001), ("United States", 2007), ("Indonesia", 1997)]
    row = filter(r -> r.country == c && coalesce(r.start_year, 0) == yr, outcomes)
    if nrow(row) > 0
        r = first(row)
        println("$c ($yr): output_loss=$(r.output_loss_pct)  fiscal_cost=$(r.fiscal_cost_pct_gdp)  npls=$(r.peak_npls_pct)  debt_increase=$(r.public_debt_increase_pct)")
    else
        println("$c ($yr): not found")
    end
end


Argentina (2001): output_loss=70.9741988  fiscal_cost=9.6  npls=20.1  debt_increase=81.88721
United States (2007): output_loss=30.003834  fiscal_cost=4.5  npls=4.99  debt_increase=21.851480303845104
Indonesia (1997): output_loss=69.0195015  fiscal_cost=56.8  npls=32.5  debt_increase=67.56022


In [23]:
# # Write Arrow with lv_ prefix
# for df in [crisis_panel, outcomes]
#     for col in names(df)
#         if nonmissingtype(eltype(df[!, col])) <: AbstractString
#             df[!, col] = passmissing(String).(df[!, col])
#         end
#     end
# end

# Arrow.write("data/lv_crisis_panel.arrow", crisis_panel)
# println("  → data/lv_crisis_panel.arrow  ($(round(filesize("data/lv_crisis_panel.arrow") / 1024, digits=1)) KB)")

# Arrow.write("data/lv_outcomes.arrow", outcomes)
# println("  → data/lv_outcomes.arrow  ($(round(filesize("data/lv_outcomes.arrow") / 1024, digits=1)) KB)")


  → data/lv_crisis_panel.arrow  (20.6 KB)
  → data/lv_outcomes.arrow  (21.1 KB)
